# Brazilian E-Commerce Dashboard Preparation

## Notebook 3

---

### Project Overview

This notebook prepares all analytical datasets required for the Power BI dashboard.

Unlike Notebook 2, this notebook does not perform business analysis. Instead, it transforms analytical data into dashboard-ready datasets that can be directly imported into Microsoft Power BI.

# Import Libraries

In [4]:
import warnings
warnings.filterwarnings("ignore")

import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format","{:,.2f}".format)

In [5]:
df = pd.read_csv("../data/processed/olist_analytical_dataset.csv")

In [6]:
datetime_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_year_month"
]

for col in datetime_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

In [7]:
df.head()

,order_id,customer_unique_id,customer_city,customer_state,seller_city,seller_state,product_category_name_english,payment_type,payment_installments,payment_value,price,freight_value,order_value,review_score,review_category,delivery_time,delivery_delay,delivery_status,purchase_year,purchase_month,purchase_weekday,purchase_hour,purchase_quarter,order_year_month
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,maua,SP,housewares,credit_card,1.00,18.12,29.99,8.72,38.71,4.00,Good,8.00,-8.00,On Time,2017,10,Monday,10,Q4,2017-10-01
1,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,maua,SP,housewares,voucher,1.00,2.00,29.99,8.72,38.71,4.00,Good,8.00,-8.00,On Time,2017,10,Monday,10,Q4,2017-10-01
2,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,maua,SP,housewares,voucher,1.00,18.59,29.99,8.72,38.71,4.00,Good,8.00,-8.00,On Time,2017,10,Monday,10,Q4,2017-10-01
3,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,barreiras,BA,belo horizonte,SP,perfumery,boleto,1.00,141.46,118.70,22.76,141.46,4.00,Good,13.00,-6.00,On Time,2018,7,Tuesday,20,Q3,2018-07-01
4,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,guariba,SP,auto,credit_card,3.00,179.12,159.90,19.22,179.12,5.00,Excellent,9.00,-18.00,On Time,2018,8,Wednesday,8,Q3,2018-08-01


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  object        
 1   customer_unique_id             119143 non-null  object        
 2   customer_city                  119143 non-null  object        
 3   customer_state                 119143 non-null  object        
 4   seller_city                    118310 non-null  object        
 5   seller_state                   118310 non-null  object        
 6   product_category_name_english  116576 non-null  object        
 7   payment_type                   119140 non-null  object        
 8   payment_installments           119140 non-null  float64       
 9   payment_value                  119140 non-null  float64       
 10  price                          118310 non-null  float64       
 11  

# Create Dashboard Folder

In [9]:
dashboard_path = "../data/dashboard"

os.makedirs(
    dashboard_path,
    exist_ok=True
)

print("Dashboard folder is ready.")

Dashboard folder is ready.


# Executive KPI Dataset

The following dataset summarizes the most important KPIs that will be displayed on the Executive Dashboard page.

In [10]:
executive_kpi = pd.DataFrame({

    "Total_Revenue":[
        df["payment_value"].sum()
    ],

    "Total_Orders":[
        df["order_id"].nunique()
    ],

    "Total_Customers":[
        df["customer_unique_id"].nunique()
    ],

    "Average_Order_Value":[
        df.groupby("order_id")["payment_value"].sum().mean()
    ],

    "Average_Review_Score":[
        df["review_score"].mean()
    ],

    "Average_Delivery_Time":[
        df["delivery_time"].mean()
    ]

})

executive_kpi

,Total_Revenue,Total_Orders,Total_Customers,Average_Order_Value,Average_Review_Score,Average_Delivery_Time
0,"20,579,664.01",99441,96096,206.95,4.02,12.02


In [11]:
executive_kpi.to_csv(
    "../data/dashboard/executive_kpi.csv",
    index=False
)

# Monthly Revenue Dataset

This dataset will be used to visualize monthly revenue trends on the Sales Dashboard.

In [12]:
monthly_revenue = (
    df.groupby(
        "order_year_month",
        as_index=False
    )

    .agg(
        Revenue=("payment_value","sum")
    )

    .sort_values(
        "order_year_month"
    )
)

monthly_revenue

,order_year_month,Revenue
0,2016-09-01,388.47
1,2016-10-01,"76,559.05"
2,2016-12-01,19.62
3,2017-01-01,"190,806.27"
4,2017-02-01,"351,848.13"
5,2017-03-01,"547,769.84"
6,2017-04-01,"512,126.52"
7,2017-05-01,"737,425.31"
8,2017-06-01,"613,777.41"
9,2017-07-01,"749,242.84"


In [13]:
monthly_revenue.to_csv(
    "../data/dashboard/monthly_revenue.csv",
    index=False
)

# Monthly Orders Dataset

This dataset summarizes the number of unique orders for each month and will be used to visualize monthly order trends.

In [14]:
monthly_orders = (
    df.groupby("order_year_month", as_index=False)
      .agg(
          Orders=("order_id", "nunique")
      )
      .sort_values("order_year_month")
)

monthly_orders

,order_year_month,Orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [15]:
monthly_orders.to_csv(
    "../data/dashboard/monthly_orders.csv",
    index=False
)

# Monthly Customers Dataset

This dataset summarizes the number of active customers each month.

In [16]:
monthly_customers = (
    df.groupby("order_year_month", as_index=False)
      .agg(
          Customers=("customer_unique_id", "nunique")
      )
      .sort_values("order_year_month")
)

monthly_customers

,order_year_month,Customers
0,2016-09-01,4
1,2016-10-01,321
2,2016-12-01,1
3,2017-01-01,765
4,2017-02-01,1755
5,2017-03-01,2642
6,2017-04-01,2372
7,2017-05-01,3625
8,2017-06-01,3180
9,2017-07-01,3947


In [17]:
monthly_customers.to_csv(
    "../data/dashboard/monthly_customers.csv",
    index=False
)

# Revenue Growth Dataset

This dataset calculates month-over-month revenue growth.

In [18]:
monthly_growth = monthly_revenue.copy()

monthly_growth["Revenue Growth (%)"] = (
    monthly_growth["Revenue"]
    .pct_change()
    .mul(100)
    .round(2)
)

monthly_growth

,order_year_month,Revenue,Revenue Growth (%)
0,2016-09-01,388.47,NaN
1,2016-10-01,"76,559.05","19,607.84"
2,2016-12-01,19.62,-99.97
3,2017-01-01,"190,806.27","972,409.02"
4,2017-02-01,"351,848.13",84.40
5,2017-03-01,"547,769.84",55.68
6,2017-04-01,"512,126.52",-6.51
7,2017-05-01,"737,425.31",43.99
8,2017-06-01,"613,777.41",-16.77
9,2017-07-01,"749,242.84",22.07


In [19]:
monthly_growth.to_csv(
    "../data/dashboard/monthly_growth.csv",
    index=False
)

# Average Order Value Dataset

This dataset contains the Average Order Value (AOV) for dashboard visualization.

In [20]:
aov_dataset = (
    df.groupby("order_id", as_index=False)
      .agg(
          Order_Value=("payment_value", "sum")
      )
)

aov_dataset

,order_id,Order_Value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04
...,...,...
99436,fffc94f6ce00a00581880bf54a75a037,343.40
99437,fffcd46ef2263f404302a634eb57f7eb,386.53
99438,fffce4705a9662cd70adb13d4a31832d,116.85
99439,fffe18544ffabc95dfada21779c9644f,64.71


In [21]:
aov_dataset.to_csv(
    "../data/dashboard/aov_dataset.csv",
    index=False
)

# Payment Method Dataset

This dataset summarizes revenue and order counts by payment method.

In [22]:
payment_summary = (
    df.groupby("payment_type", as_index=False)
      .agg(
          Revenue=("payment_value", "sum"),
          Orders=("order_id", "nunique")
      )
      .sort_values("Revenue", ascending=False)
)

payment_summary

,payment_type,Revenue,Orders
1,credit_card,"15,775,450.54",76505
0,boleto,"4,110,920.74",19784
4,voucher,"435,917.84",3866
2,debit_card,"257,374.89",1528
3,not_defined,0.00,3


In [23]:
payment_summary.to_csv(
    "../data/dashboard/payment_summary.csv",
    index=False
)

# Weekday Dataset

This dataset summarizes customer purchasing behavior by day of the week.

In [24]:
weekday_summary = (
    df.groupby("purchase_weekday", as_index=False)
      .agg(
          Revenue=("payment_value", "sum"),
          Orders=("order_id", "nunique")
      )
)

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_summary["purchase_weekday"] = pd.Categorical(
    weekday_summary["purchase_weekday"],
    categories=weekday_order,
    ordered=True
)

weekday_summary = weekday_summary.sort_values("purchase_weekday")

weekday_summary

,purchase_weekday,Revenue,Orders
1,Monday,"3,347,786.31",16196
5,Tuesday,"3,361,564.89",15963
6,Wednesday,"3,198,612.69",15552
4,Thursday,"3,145,030.37",14761
0,Friday,"3,056,608.65",14122
2,Saturday,"2,172,171.12",10887
3,Sunday,"2,297,889.98",11960


In [25]:
weekday_summary.to_csv(
    "../data/dashboard/weekday_summary.csv",
    index=False
)

# Hourly Dataset

This dataset summarizes customer purchasing activity by hour.

In [26]:
hourly_summary = (
    df.groupby("purchase_hour", as_index=False)
      .agg(
          Revenue=("payment_value", "sum"),
          Orders=("order_id", "nunique")
      )
      .sort_values("purchase_hour")
)

hourly_summary

,purchase_hour,Revenue,Orders
0,0,"474,801.20",2394
1,1,"210,304.96",1170
2,2,"92,701.68",510
3,3,"51,356.55",272
4,4,"36,004.52",206
5,5,"30,422.81",188
6,6,"82,144.98",502
7,7,"226,365.03",1231
8,8,"602,806.69",2967
9,9,"1,108,161.14",4785


In [27]:
hourly_summary.to_csv(
    "../data/dashboard/hourly_summary.csv",
    index=False
)

# State Summary Dataset

This dataset summarizes customer distribution and revenue by state for the Geography Dashboard.

In [28]:
state_summary = (
    df.groupby("customer_state", as_index=False)
      .agg(
          Customers=("customer_unique_id", "nunique"),
          Orders=("order_id", "nunique"),
          Revenue=("payment_value", "sum")
      )
      .sort_values("Revenue", ascending=False)
)

state_summary

,customer_state,Customers,Orders,Revenue
25,SP,40302,41746,"7,726,078.35"
18,RJ,12384,12852,"2,795,615.67"
10,MG,11259,11635,"2,351,221.09"
22,RS,5277,5466,"1,160,175.66"
17,PR,4882,5045,"1,079,795.49"
4,BA,3277,3380,"805,070.98"
23,SC,3534,3637,"801,276.45"
8,GO,1952,2020,"520,481.65"
6,DF,2075,2140,"438,095.32"
7,ES,1964,2033,"408,611.64"


In [29]:
state_summary.to_csv(
    "../data/dashboard/state_summary.csv",
    index=False
)

# City Summary Dataset

This dataset summarizes customer distribution and revenue by city.

In [30]:
city_summary = (
    df.groupby("customer_city", as_index=False)
      .agg(
          Customers=("customer_unique_id", "nunique"),
          Orders=("order_id", "nunique"),
          Revenue=("payment_value", "sum")
      )
      .sort_values("Revenue", ascending=False)
)

city_summary

,customer_city,Customers,Orders,Revenue
3597,sao paulo,14984,15540,"2,901,789.67"
3155,rio de janeiro,6620,6882,"1,581,736.07"
453,belo horizonte,2672,2773,"509,165.81"
558,brasilia,2069,2131,"435,971.02"
1143,curitiba,1465,1521,"333,582.09"
...,...,...,...,...
3403,santo antonio do rio abaixo,1,1,24.23
3793,tamboara,1,1,24.09
1957,jenipapo de minas,1,1,22.58
2928,polo petroquimico de triunfo,1,1,20.70


In [31]:
city_summary.to_csv(
    "../data/dashboard/city_summary.csv",
    index=False
)

# RFM Summary Dataset

This dataset summarizes customer segmentation using the RFM model.

In [32]:
snapshot_date = (
    df["order_year_month"].max() +
    pd.Timedelta(days=1)
)

rfm = (
    df.groupby("customer_unique_id")
      .agg(
          Recency=(
              "order_year_month",
              lambda x: (snapshot_date - x.max()).days
          ),
          Frequency=("order_id", "nunique"),
          Monetary=("payment_value", "sum")
      )
      .reset_index()
)

In [33]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)

In [34]:
def customer_segment(row):

    score = (
        int(row["R_Score"])
        + int(row["F_Score"])
        + int(row["M_Score"])
    )

    if score >= 13:
        return "Champions"

    elif score >= 10:
        return "Loyal Customers"

    elif score >= 8:
        return "Potential Loyalists"

    elif score >= 6:
        return "Need Attention"

    else:
        return "At Risk"

rfm["Segment"] = rfm.apply(
    customer_segment,
    axis=1
)

In [35]:
rfm_summary = (
    rfm.groupby("Segment", as_index=False)
       .agg(
           Customers=("customer_unique_id", "count"),
           Avg_Monetary=("Monetary", "mean"),
           Avg_Frequency=("Frequency", "mean")
       )
)

rfm_summary

,Segment,Customers,Avg_Monetary,Avg_Frequency
0,At Risk,7855,57.03,1.00
1,Champions,9084,475.13,1.20
2,Loyal Customers,32913,275.47,1.04
3,Need Attention,18563,103.08,1.00
4,Potential Loyalists,27681,174.69,1.01


In [36]:
rfm_summary.to_csv(
    "../data/dashboard/rfm_summary.csv",
    index=False
)

# Pareto Summary Dataset

This dataset measures customer revenue concentration based on the Pareto principle.

In [37]:
customer_revenue = (
    df.groupby("customer_unique_id", as_index=False)
      .agg(
          Revenue=("payment_value", "sum")
      )
      .sort_values("Revenue", ascending=False)
)

customer_revenue["Cumulative Revenue"] = (
    customer_revenue["Revenue"].cumsum()
)

customer_revenue["Revenue Share (%)"] = (
    customer_revenue["Cumulative Revenue"]
    / customer_revenue["Revenue"].sum()
    * 100
)

customer_revenue["Customer Share (%)"] = (
    np.arange(1, len(customer_revenue)+1)
    / len(customer_revenue)
    * 100
)

customer_revenue.head()

,customer_unique_id,Revenue,Cumulative Revenue,Revenue Share (%),Customer Share (%)
3826,0a0a92112bd4c708ca5fde585afaa872,"109,312.64","109,312.64",0.53,0.00
39720,698e1cf81d01a3d389d96145f7fa6df8,"45,256.00","154,568.64",0.75,0.00
73664,c402f431464c72e27330a67f7b94d4fb,"44,048.00","198,616.64",0.97,0.00
24121,4007669dec559734d6f53e029e360987,"36,489.24","235,105.88",1.14,0.00
90000,ef8d54b3797ea4db1d63f0ced6a906e9,"30,186.00","265,291.88",1.29,0.01


In [38]:
customer_revenue.to_csv(
    "../data/dashboard/pareto_summary.csv",
    index=False
)

# Customer Lifetime Value Dataset

This dataset summarizes revenue generated by each customer.

In [39]:
clv_summary = (
    df.groupby("customer_unique_id", as_index=False)
      .agg(
          Total_Revenue=("payment_value", "sum"),
          Total_Orders=("order_id", "nunique")
      )
)

clv_summary["Average_Order_Value"] = (
    clv_summary["Total_Revenue"]
    / clv_summary["Total_Orders"]
)

clv_summary.head()

,customer_unique_id,Total_Revenue,Total_Orders,Average_Order_Value
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,1,27.19
2,0000f46a3911fa3c0805444483337064,86.22,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89,1,196.89


In [40]:
clv_summary.to_csv(
    "../data/dashboard/clv_summary.csv",
    index=False
)

# Export Summary

All dashboard datasets have been successfully created and exported to the `data/dashboard` folder.

These datasets are now ready to be imported into Microsoft Power BI for dashboard development.

In [41]:
import os

files = sorted(os.listdir("../data/dashboard"))

print("Dashboard datasets created successfully:\n")

for file in files:
    print(file)

Dashboard datasets created successfully:

aov_dataset.csv
city_summary.csv
clv_summary.csv
executive_kpi.csv
hourly_summary.csv
monthly_customers.csv
monthly_growth.csv
monthly_orders.csv
monthly_revenue.csv
pareto_summary.csv
payment_summary.csv
rfm_summary.csv
state_summary.csv
weekday_summary.csv
